# Import 

In [1]:
import pandas as pd
import numpy as np
from os.path import exists


In [2]:
import sys
sys.path.append('../../../')
sys.path.append('../../')

from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

In [3]:
from map_bigg_04 import *

# Parameters

In [4]:
too_general = {'1.1.1.1', '1.1.1.100', '1.1.1.122', '1.1.1.192',
               '1.1.1.2', '1.1.1.21', '1.1.1.90', '1.1.3.13', '1.2.1.3',
               '1.2.1.4', '1.3.1.9', '1.3.8.7', '1.3.8.8', '2.3.1.180',
               '2.3.1.181', '2.3.1.300', '2.3.1.38', '2.3.1.39', '2.3.1.40',
               '2.3.1.41', '2.3.1.84', '2.3.1.85', '2.6.1.57', '2.7.1.1',
               '2.7.11.1', '2.7.11.22', '2.7.8.7', '3.1.1.1', '3.1.1.2', '3.1.1.3',
               '3.1.1.5', '3.1.2.14', '3.1.2.20', '3.1.3.1', '3.1.3.106', '3.1.3.2',
               '3.1.3.4', '3.1.3.5', '3.5.2.6', '4.1.1.1', '4.2.1.59', '4.4.1.13',
               '5.3.3.14', '6.2.1.2', '6.2.1.20', '6.2.1.3', '6.2.1.47'}

dir_data_metanetx = './metanetx/'

# Files

## Input

In [5]:
(36061, 6)

(36061, 6)

In [6]:
lista = pd.read_json('Gene_modification_Clasification.json')

In [7]:
file_path = f'{dir_data_metanetx}/rxns_table.json'


## Output

# ------

In [8]:
organism_specific_rxns = None

In [9]:
BIGG_rxn_info, rxn_prop = get_tables(dir_data_metanetx)

In [10]:
# Example usage:
model1 = cobra.io.load_json_model('/Users/elisamarquez/Downloads/e_coli_core.json')
model2 = cobra.io.load_json_model('/Users/elisamarquez/Downloads/iJO1366.json')

# Apply filters sequentially
df_filtered1 = filter_df_by_model(BIGG_rxn_info, model1)
df_filtered2 = filter_df_by_model(df_filtered1, model2)

# Now merge all same MNX_ID entries by concatenating their XREFs
df_final = merge_by_mnx(df_filtered2)

Set parameter Username
Academic license - for non-commercial use only - expires 2026-01-15


In [11]:
df_final.head()

,MNX_ID,XREF
0,MNXR02,EX_h_e
1,MNXR03,Htex
2,MNXR100000,GALNACT5g
3,MNXR100001,GALNTg;NAGA2ly
4,MNXR100004,GALR1TRA1


In [12]:
BIGG_rxn_info.shape

(28160, 4)

In [13]:
df_filtered2.shape

(25920, 4)

In [14]:
BIGG_rxn_info = df_final

In [15]:
bigg_rxns = rxn_prop[rxn_prop.XREF.str.startswith('bigg')]

In [16]:
bigg_rxns = bigg_rxns.dropna(subset=['EC_number'])

In [17]:
rxn_prop = rxn_prop.dropna(subset=['EC_number'])

# TEST WITH BIGG REACTIONS

In [18]:
bigg_rxns = rxn_prop

In [19]:
bigg_rxns.head()

,MNX_ID,MNX_Formula,XREF,EC_number,Evidence,Description
15,MNXR100024,1 MNXM1@MNXD1 + 1 MNXM37@MNXD1 + 1 MNXM40333@M...,rh:16169,6.3.1.2,B,NaN
19,MNXR100030,1 MNXM37@MNXD1 + 1 WATER@MNXD1 = 1 MNXM729302@...,rh:15889,1.4.1.13;1.4.1.14;1.4.7.1;2.4.2;2.4.2.14;2.6.1...,B,NaN
36,MNXR100060,1 MNXM10@MNXD1 + 2 MNXM1@MNXD1 + 1 MNXM222@MNX...,rh:20001,1.2.1;1.2.1.19;1.2.1.21;1.2.1.3;1.2.1.4;1.2.1....,B,NaN
39,MNXR100063,1 MNXM1048@MNXD1 = 1 MNXM1108250@MNXD1,rh:32239,5.5.1.19,B,NaN
40,MNXR100064,1 MNXM1102150@MNXD1 + 1 MNXM1277@MNXD1 + 1 MNX...,seedR:rxn09498,2.1.2.10,B,NaN


In [20]:
test_list3_genes = lista
test_list3_genes.EC_number = test_list3_genes.EC_number.apply(set)

In [21]:
test_list3_genes.shape

(14434, 4)

In [22]:
test_list3_genes.head()

,Paper_ID,Gene,EC_number,Classification
0,3,pyk,{2.7.1.40},Positive
1,5,buk,{2.7.2.7},Positive
2,5,ptb,{2.3.1.19},Positive
3,5,phaE,"{6.2.1.30, 4.2.1.17}",Positive
4,10,phbB,{1.1.1.36},Positive


In [23]:
t_list3_exploded = test_list3_genes.explode('EC_number')

In [24]:
        rxns_test = pd.merge(
            t_list3_exploded, bigg_rxns,
            on='EC_number', suffixes=('_cosa', '_bigg'), how='outer'
        )


In [25]:
rxns_test.head()

,Paper_ID,Gene,EC_number,Classification,MNX_ID,MNX_Formula,XREF,Evidence,Description
0,3.0,pyk,2.7.1.40,Positive,MNXR103371,1 MNXM1@MNXD1 + 1 MNXM40333@MNXD1 + 1 MNXM73@M...,rh:18157,B,NaN
1,3.0,pyk,2.7.1.40,Positive,MNXR129735,1 MNXM1@MNXD1 + 1 MNXM411@MNXD1 + 1 MNXM73@MNX...,rh:56968,B,NaN
2,3.0,pyk,2.7.1.40,Positive,MNXR129753,1 MNXM163339@MNXD1 + 1 MNXM73@MNXD1 = 1 MNXM16...,sabiorkR:10478,NaN,NaN
3,3.0,pyk,2.7.1.40,Positive,MNXR132739,1 MNXM1103718@MNXD1 + 1 MNXM23@MNXD1 = 1 MNXM4...,sabiorkR:55,NaN,NaN
4,3.0,pyk,2.7.1.40,Positive,MNXR133519,1 MNXM165392@MNXD1 + 1 MNXM40333@MNXD1 = 1 MNX...,sabiorkR:7849,NaN,NaN


In [26]:


bigg_rxns["EC_number_clean"] = bigg_rxns["EC_number"].apply(clean_bigg_ecs)

In [27]:
bigg_rxns.head()

,MNX_ID,MNX_Formula,XREF,EC_number,Evidence,Description,EC_number_clean
15,MNXR100024,1 MNXM1@MNXD1 + 1 MNXM37@MNXD1 + 1 MNXM40333@M...,rh:16169,6.3.1.2,B,NaN,{6.3.1.2}
19,MNXR100030,1 MNXM37@MNXD1 + 1 WATER@MNXD1 = 1 MNXM729302@...,rh:15889,1.4.1.13;1.4.1.14;1.4.7.1;2.4.2;2.4.2.14;2.6.1...,B,NaN,"{6.3.4.2, 3.5.1.2, 1.4.7.1, 6.3.5.7, 2.6.1.85,..."
36,MNXR100060,1 MNXM10@MNXD1 + 2 MNXM1@MNXD1 + 1 MNXM222@MNX...,rh:20001,1.2.1;1.2.1.19;1.2.1.21;1.2.1.3;1.2.1.4;1.2.1....,B,NaN,"{1.2.1.21, 1.2.1.69, 1.2.1.4, 1.2.1.19, 1.2.1...."
39,MNXR100063,1 MNXM1048@MNXD1 = 1 MNXM1108250@MNXD1,rh:32239,5.5.1.19,B,NaN,{5.5.1.19}
40,MNXR100064,1 MNXM1102150@MNXD1 + 1 MNXM1277@MNXD1 + 1 MNX...,seedR:rxn09498,2.1.2.10,B,NaN,{2.1.2.10}


In [28]:
import pandas as pd

# --- 1. Explode EC lists ---
genes = (
    test_list3_genes
    .explode("EC_number")
    .rename(columns={"EC_number": "EC"})
)
rxns = (
    bigg_rxns
    .explode("EC_number_clean")
    .rename(columns={"EC_number_clean": "EC"})
)

# --- 2. Merge on EC number ---
merged = genes.merge(rxns, on="EC", how="inner")

# --- 3. Aggregate matches ---
matched_df = (
    merged.groupby(["Gene", "Paper_ID", "MNX_ID", "XREF"], as_index=False)
          .agg(Matching_ECs=("EC", lambda x: set(x)))
)

# --- 4. Prepare lookups ---
gene_ec_lookup = test_list3_genes.set_index("Gene")["EC_number"].to_dict()
rxn_ec_lookup = bigg_rxns.set_index("MNX_ID")["EC_number_clean"].to_dict()


# --- 5. Compute all metrics ---
extra = matched_df.apply(compute_all_metrics, axis=1,
                         args = (too_general,gene_ec_lookup,rxn_ec_lookup,))
matched_df = pd.concat([matched_df, extra], axis=1)

# --- 6. Final tidy-up ---
matched_df = matched_df.rename(columns={
    "MNX_ID": "Reaction_ID",
    "XREF": "BIGG_ID"
})

matched_df = matched_df[
    ["Gene", "Paper_ID", "Reaction_ID", "BIGG_ID",
     "Matching_ECs", "Unmatched_Gene_ECs", "Unmatched_Reaction_ECs",
     "Match_Count", "Gene_EC_Total", "Reaction_EC_Total",
     "Fraction_Matched", "Corrected_Fraction_Matched",
     "Reaction_Fraction_Matched", "Reaction_Corrected_Fraction_Matched"]
]

matched_df.head()


,Gene,Paper_ID,Reaction_ID,BIGG_ID,Matching_ECs,Unmatched_Gene_ECs,Unmatched_Reaction_ECs,Match_Count,Gene_EC_Total,Reaction_EC_Total,Fraction_Matched,Corrected_Fraction_Matched,Reaction_Fraction_Matched,Reaction_Corrected_Fraction_Matched
0,AAE,8268,MNXR117401,seedR:rxn42506,{6.2.1.2},{6.2.1.1},{6.2.1.3},1,2,2,0.5,0.5,0.5,1.0
1,AAE,8268,MNXR132150,sabiorkR:13579,{6.2.1.1},{6.2.1.2},{},1,2,1,0.5,1.0,1.0,1.0
2,AAE,8268,MNXR132151,sabiorkR:13580,{6.2.1.1},{6.2.1.2},{},1,2,1,0.5,1.0,1.0,1.0
3,AAE,8268,MNXR132152,sabiorkR:13581,{6.2.1.1},{6.2.1.2},{},1,2,1,0.5,1.0,1.0,1.0
4,AAE,8268,MNXR132153,sabiorkR:13582,{6.2.1.1},{6.2.1.2},{},1,2,1,0.5,1.0,1.0,1.0


In [29]:
things_in_bigg = matched_df.loc[matched_df.Reaction_ID.isin(BIGG_rxn_info.MNX_ID)]

In [30]:
merged_bigg = things_in_bigg.merge(
    BIGG_rxn_info[["MNX_ID", "XREF"]],   # select only the needed columns
    left_on="Reaction_ID",               # match Reaction_ID from things_in_bigg
    right_on="MNX_ID",                   # match MNX_ID from BIGG_rxn_info
    how="left"                           # keep all rows from things_in_bigg
)

# Optional cleanup: drop redundant MNX_ID column
merged_bigg = merged_bigg.drop(columns=["MNX_ID"])


In [31]:
merged_bigg

,Gene,Paper_ID,Reaction_ID,BIGG_ID,Matching_ECs,Unmatched_Gene_ECs,Unmatched_Reaction_ECs,Match_Count,Gene_EC_Total,Reaction_EC_Total,Fraction_Matched,Corrected_Fraction_Matched,Reaction_Fraction_Matched,Reaction_Corrected_Fraction_Matched,XREF
0,AAE,8268,MNXR147173,rh:46172,"{6.2.1.2, 6.2.1.1}",{},"{6.2.1.17, 6.2.1.3}",2,2,4,1.0,1.0,0.500,0.667,FACOAL40im;HMR_0156
1,AAE,8268,MNXR153095,rh:10132,{6.2.1.2},{6.2.1.1},"{6.2.1.25, 6.2.1.33}",1,2,3,0.5,0.5,0.333,0.333,BCOALIG;BZCOAFm
2,AAE,8268,MNXR189030,keggR:R00236,{6.2.1.1},{6.2.1.2},{},1,2,1,0.5,1.0,1.000,1.000,r0068
3,AAE,8268,MNXR189032,rh:58136,{6.2.1.1},{6.2.1.2},{},1,2,1,0.5,1.0,1.000,1.000,r0097
4,AAE,8268,MNXR189036,keggR:R00926,{6.2.1.1},{6.2.1.2},{6.2.1.17},1,2,2,0.5,1.0,0.500,0.500,r0220;r0220_1;r0221;r0221_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72576,zwf2,3017,MNXR153708,rh:38215,{1.1.1.49},{},"{1.1.1.363, 1.1.1.388, 1.1.1.47}",1,1,4,1.0,1.0,0.250,0.250,G6PDH1;G6PDH1er
72577,zwf2,3017,MNXR191105,keggR:R02736,{1.1.1.49},{},{1.1.1.363},1,1,2,1.0,1.0,0.500,0.500,G6PBDH;G6PBDHh;G6PDHg
72578,zwf2,3017,MNXR192434,rh:15841,{1.1.1.49},{},"{1.1.1.363, 1.1.1.47}",1,1,3,1.0,1.0,0.333,0.333,G6PDH2r
72579,zwf2,3017,MNXR192435,seedR:rxn15072,{1.1.1.49},{},{},1,1,1,1.0,1.0,1.000,1.000,G6PADH;G6PADHh


In [32]:
def check_model(xref):
    if xref in model1.reactions:
        return(2)
    elif xref in model2.reactions:
        return(1)
    else:
        return(False)

In [33]:
merged_bigg['Presence'] = merged_bigg.XREF.apply(check_model)

In [34]:
df = merged_bigg.copy()


In [35]:

# --- 1. Compute a "specific match score" ---
def compute_specific_match_score(ec_set):
    if not ec_set:
        return 0
    non_general = [ec for ec in ec_set if ec not in too_general]
    return len(non_general)



df["Specific_Match_Score"] = df["Matching_ECs"].apply(compute_specific_match_score)

# --- 2. Sort by your new priority rules ---
df = df.sort_values(
    by=[
        "Gene",
        "Paper_ID",
        "Specific_Match_Score", 
        "Match_Count",
        "Presence",
            # NEW: prioritize specific matches
        "Fraction_Matched",
        "Corrected_Fraction_Matched",
        "Reaction_Corrected_Fraction_Matched",
        
    ],
    ascending=[True, True, False, False, False, False, False,False]
)


In [36]:

df.sort_values('Paper_ID').head()

,Gene,Paper_ID,Reaction_ID,BIGG_ID,Matching_ECs,Unmatched_Gene_ECs,Unmatched_Reaction_ECs,Match_Count,Gene_EC_Total,Reaction_EC_Total,Fraction_Matched,Corrected_Fraction_Matched,Reaction_Fraction_Matched,Reaction_Corrected_Fraction_Matched,XREF,Presence,Specific_Match_Score
63145,pyk,3,MNXR188730,rh:56952,{2.7.1.40},{},{},1,1,1,1.0,1.0,1.0,1.0,PYK4,False,1
63147,pyk,3,MNXR95613,rh:30791,{2.7.1.40},{},{},1,1,1,1.0,1.0,1.0,1.0,AGPOP;AGPOPm,False,1
63142,pyk,3,MNXR103371,rh:18157,{2.7.1.40},{},{},1,1,1,1.0,1.0,1.0,1.0,PYK,2,1
63141,pyk,3,MNXR103140,rh:11364,{2.7.1.40},{},{2.7.9.2},1,1,2,1.0,1.0,0.5,0.5,PPS,2,1
63143,pyk,3,MNXR188179,rh:30295,{2.7.1.40},{},{},1,1,1,1.0,1.0,1.0,1.0,GTPOPm;PYK3,False,1


In [37]:
df.to_json('Gene_X_BIGG.json')